In [50]:
import pandas as pd
import json

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [ ]:
df = pd.read_csv("data/es_cereal_46_gpt_preds_subsentences.csv")

rows = list()
for idx, row in df.iterrows():
    preds = json.loads(row["preds"])
    for token in preds:
        temp = dict()
        temp["sent"] = row["sents"]
        for k in token.keys():
            temp[k] = token[k]
        rows.append(temp)

preds_df = pd.DataFrame(rows)

preds_df

In [ ]:
columns = ["sent", "word", "manual_fee", "manual_frame"]
manual_df = pd.read_csv("data/spanish_fees.csv", header=0, names=columns)

manual_df

In [ ]:
combined_df = pd.concat([preds_df, manual_df[["manual_fee", "manual_frame"]]], axis=1)

combined_df

Total non-matches, including fee=False:

In [ ]:
combined_df[combined_df["frame"]!=combined_df["manual_frame"]]

In [55]:
print(f"""Complete match at {combined_df[combined_df["frame"]==combined_df["manual_frame"]].shape[0] / combined_df.shape[0] * 100}%""")

Complete match at 66.634545892556%


Total non-matches, excluding fee=False:

In [ ]:
combined_df[(combined_df["frame"]!=combined_df["manual_frame"])&(combined_df["fee"]!=False)&(combined_df["manual_fee"]!=False)]

In [57]:
print(f"""Only fee=True complete match at {combined_df[(combined_df["frame"]==combined_df["manual_frame"])&(combined_df["fee"]!=False)&(combined_df["manual_fee"]!=False)].shape[0] / combined_df[(combined_df["fee"]!=False)&(combined_df["manual_fee"]!=False)].shape[0] * 100}%""")

Only fee=True complete match at 49.05183312262958%


Percentage of To_Review FEEs:

In [58]:
print(f"""For automatic annotation: {combined_df[combined_df["frame"]=="To_Review"].shape[0] / combined_df[combined_df["fee"]==True].shape[0] * 100}%""")

For automatic annotation: 0.6165228113440198%


In [59]:
print(f"""For manual annotation: {combined_df[combined_df["manual_frame"]=="To_Review"].shape[0] / combined_df[combined_df["manual_fee"]==True].shape[0] * 100}%""")

For manual annotation: 2.9702970297029703%


Percentage of manual FEEs:

In [60]:
print(f"""For manual annotation: {combined_df[combined_df["manual_fee"]==True].shape[0] / combined_df.shape[0] * 100}%""")

For manual annotation: 51.09612141652614%


In [61]:
print(f"""Accuracy: {accuracy_score(combined_df["manual_frame"], combined_df["frame"])}""")

Accuracy: 0.6663454589255601


In [62]:
print(f"""Micro metrics:\n
Precision: {precision_score(combined_df["manual_frame"], combined_df["frame"], average="micro")}
Recall: {recall_score(combined_df["manual_frame"], combined_df["frame"], average="micro")}
F1-score: {f1_score(combined_df["manual_frame"], combined_df["frame"], average="micro")}
""")

Micro metrics:

Precision: 0.6663454589255601
Recall: 0.6663454589255601
F1-score: 0.6663454589255601



In [63]:
print(f"""Macro metrics:\n
Precision: {precision_score(combined_df["manual_frame"], combined_df["frame"], average="macro", zero_division=0)}
Recall: {recall_score(combined_df["manual_frame"], combined_df["frame"], average="macro", zero_division=0)}
F1-score: {f1_score(combined_df["manual_frame"], combined_df["frame"], average="macro", zero_division=0)}
""")

Macro metrics:

Precision: 0.32919177231396557
Recall: 0.2992398400924534
F1-score: 0.28263220397312394



In [64]:
print(f"""Weighted metrics:\n
Precision: {precision_score(combined_df["manual_frame"], combined_df["frame"], average="weighted", zero_division=0)}
Recall: {recall_score(combined_df["manual_frame"], combined_df["frame"], average="weighted", zero_division=0)}
F1-score: {f1_score(combined_df["manual_frame"], combined_df["frame"], average="weighted", zero_division=0)}
""")

Weighted metrics:

Precision: 0.683450159143957
Recall: 0.6663454589255601
F1-score: 0.6259233667347878



In [65]:
combined_df[["manual_frame"]].value_counts()

manual_frame    
-                   2030
Goal                  65
To_Review             63
Substance             54
Degree                49
                    ... 
Arrest                 1
Containers             1
Contacting             1
People_by_origin       1
Calendar_unit          1
Name: count, Length: 483, dtype: int64